# Qwen2-Audio 7B - Hallucination Detection Experiment

# Setup

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
%cd /content/drive/MyDrive/ICASSP_Hallucinaton/

/content/drive/MyDrive/ICASSP_Hallucinaton


In [3]:
%pip install transformers accelerate soundfile pandas tqdm librosa

# Data

# Main

In [7]:
import torch
import pandas as pd
import json
import os
import time
import hashlib
import pickle
from pathlib import Path
from datetime import datetime
from typing import Dict, List, Tuple, Optional, Set
from tqdm import tqdm
import librosa
# Set CUDA memory management environment variable
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

# ========================== CONFIGURATION ==========================

# Base data directory
BASE_DATA_DIR = "/content/drive/MyDrive/ICASSP_Hallucinaton/Audio_data"

# Language folders
LANGUAGE_FOLDERS = {
    'english': 'English',
    'kazakh': 'Kazakh',
    'russian': 'Russian'
}

# Models to test
MODELS = [
    "Qwen/Qwen2-Audio-7B-Instruct",
]

# Experiment types
EXPERIMENT_TYPES = ['audio', 'text']

# Classification tasks
TASKS = ['binary', 'type', 'degree']

# Prompting approaches
APPROACHES = ['direct', 'cot']

# Prompt structure version (increment when prompt structure changes to invalidate old checkpoints)
PROMPT_VERSION = 2  # v1 = combined 3-task prompt, v2 = separated single-task prompts

# ---- BATCH SIZE FOR TEXT EXPERIMENTS ----
# Adjust based on GPU memory. Start with 4, increase if no OOM.
# Audio experiments still run with batch_size=1 due to variable-length audio padding.
# Qwen2-Audio-7B uses ~14GB, so on a 24GB GPU try 4; on 16GB try 2.
TEXT_BATCH_SIZE = 8

# Output directory structure
OUTPUT_DIR = "hallucination_results"
CHECKPOINT_DIR = "checkpoints"

# ========================== PROMPTS ==========================

# System prompt for audio processing
SYSTEM_PROMPT = "You are Qwen, a virtual human developed by the Qwen Team, Alibaba Group, capable of perceiving auditory and visual inputs, as well as generating text and speech."

# --- TASK 1: Binary hallucination detection ---

AUDIO_BINARY_DIRECT_PROMPT = """You are an expert assistant specialized in analyzing audio content and detecting hallucinations.

Your task is to determine whether there are any hallucinations in the audio content.

A hallucination is any statement that contains factual contradictions, fabricated details, or contextual inconsistencies relative to the actual audio content.

Given the audio content, determine:
Is there a hallucination? (yes/no)

Output format: {{"binary": "yes/no"}}"""

AUDIO_BINARY_COT_PROMPT = """You are an expert assistant specialized in analyzing audio content and detecting hallucinations.

Your task is to determine whether there are any hallucinations in the audio content.

A hallucination is any statement that contains factual contradictions, fabricated details, or contextual inconsistencies relative to the actual audio content.

Given the audio content, think step-by-step about whether hallucinations are present:

Then provide your final answer:
Is there a hallucination? (yes/no)

Output format: {{"binary": "yes/no"}}"""

# --- TASK 2: Hallucination type classification ---

AUDIO_TYPE_DIRECT_PROMPT = """You are an expert assistant specialized in analyzing audio content and classifying hallucination types.

Your task is to identify the type of hallucination present in the audio content.

Hallucination Types:
- Factual Contradiction: Statements that directly conflict with known facts or information provided in the audio content.
- Factual Fabrication: Insertion of fabricated yet plausible-sounding details not grounded in the audio content.
- Contextual Inconsistency: Subtle alterations that distort the meaning, emphasis, or context of the audio content without introducing explicit factual errors.

Given the audio content, classify the hallucination type:
What type of hallucination is present? (factual_contradiction/factual_fabrication/contextual_inconsistency/none)

Output format: {{"type": "factual_contradiction/factual_fabrication/contextual_inconsistency/none"}}"""

AUDIO_TYPE_COT_PROMPT = """You are an expert assistant specialized in analyzing audio content and classifying hallucination types.

Your task is to identify the type of hallucination present in the audio content.

Hallucination Types:
- Factual Contradiction: Statements that directly conflict with known facts or information provided in the audio content.
- Factual Fabrication: Insertion of fabricated yet plausible-sounding details not grounded in the audio content.
- Contextual Inconsistency: Subtle alterations that distort the meaning, emphasis, or context of the audio content without introducing explicit factual errors.

Given the audio content, think step-by-step about what type of hallucination may be present:

Then provide your final classification:
What type of hallucination is present? (factual_contradiction/factual_fabrication/contextual_inconsistency/none)

Output format: {{"type": "factual_contradiction/factual_fabrication/contextual_inconsistency/none"}}"""

# --- TASK 3: Hallucination degree/severity classification ---

AUDIO_DEGREE_DIRECT_PROMPT = """You are an expert assistant specialized in analyzing audio content and assessing hallucination severity.

Your task is to assess the severity level of any hallucination present in the audio content.

Severity Levels:
- Mild: Subtle distortions or minor deviations that preserve the main narrative and plausibility of the audio content.
- Moderate: Noticeable inconsistencies or factual alterations that affect key details or context while maintaining partial alignment with the audio content.
- Severe: Major contradictions, fabrications, or contextual breakdowns that substantially misrepresent or conflict with the audio content's facts or intent.

Given the audio content, classify the hallucination severity:
What is the severity level? (mild/moderate/severe/none)

Output format: {{"degree": "mild/moderate/severe/none"}}"""

AUDIO_DEGREE_COT_PROMPT = """You are an expert assistant specialized in analyzing audio content and assessing hallucination severity.

Your task is to assess the severity level of any hallucination present in the audio content.

Severity Levels:
- Mild: Subtle distortions or minor deviations that preserve the main narrative and plausibility of the audio content.
- Moderate: Noticeable inconsistencies or factual alterations that affect key details or context while maintaining partial alignment with the audio content.
- Severe: Major contradictions, fabrications, or contextual breakdowns that substantially misrepresent or conflict with the audio content's facts or intent.

Given the audio content, think step-by-step about the severity of any hallucination:

Then provide your final assessment:
What is the severity level? (mild/moderate/severe/none)

Output format: {{"degree": "mild/moderate/severe/none"}}"""

# --- Text prompts (derived from audio prompts) ---

TEXT_BINARY_DIRECT_PROMPT = AUDIO_BINARY_DIRECT_PROMPT.replace("audio content", "text content")
TEXT_BINARY_COT_PROMPT = AUDIO_BINARY_COT_PROMPT.replace("audio content", "text content")
TEXT_TYPE_DIRECT_PROMPT = AUDIO_TYPE_DIRECT_PROMPT.replace("audio content", "text content")
TEXT_TYPE_COT_PROMPT = AUDIO_TYPE_COT_PROMPT.replace("audio content", "text content")
TEXT_DEGREE_DIRECT_PROMPT = AUDIO_DEGREE_DIRECT_PROMPT.replace("audio content", "text content")
TEXT_DEGREE_COT_PROMPT = AUDIO_DEGREE_COT_PROMPT.replace("audio content", "text content")

# Prompt lookup: PROMPTS[modality][task][approach]
PROMPTS = {
    'audio': {
        'binary': {'direct': AUDIO_BINARY_DIRECT_PROMPT, 'cot': AUDIO_BINARY_COT_PROMPT},
        'type':   {'direct': AUDIO_TYPE_DIRECT_PROMPT,   'cot': AUDIO_TYPE_COT_PROMPT},
        'degree': {'direct': AUDIO_DEGREE_DIRECT_PROMPT, 'cot': AUDIO_DEGREE_COT_PROMPT},
    },
    'text': {
        'binary': {'direct': TEXT_BINARY_DIRECT_PROMPT, 'cot': TEXT_BINARY_COT_PROMPT},
        'type':   {'direct': TEXT_TYPE_DIRECT_PROMPT,   'cot': TEXT_TYPE_COT_PROMPT},
        'degree': {'direct': TEXT_DEGREE_DIRECT_PROMPT, 'cot': TEXT_DEGREE_COT_PROMPT},
    }
}


# ========================== SETUP ==========================

def setup_model(model_name: str, use_flash_attn: bool = False):
    """Load and setup Qwen2-Audio model"""
    from transformers import Qwen2AudioForConditionalGeneration, AutoProcessor

    print(f"Loading model: {model_name}")

    kwargs = {
        "torch_dtype": "auto",
        "device_map": "auto"
    }

    if use_flash_attn:
        kwargs["attn_implementation"] = "flash_attention_2"

    model = Qwen2AudioForConditionalGeneration.from_pretrained(model_name, **kwargs)
    processor = AutoProcessor.from_pretrained(model_name)

    print(f"Model loaded successfully: {model_name}")
    return model, processor

# ========================== DATA LOADING ==========================

def load_transcription_data(language_folder: str, language: str, filter_hallucinated_only: bool = False) -> pd.DataFrame:
    """Load transcription file from language folder with optional hallucination filtering"""

    folder_path = Path(BASE_DATA_DIR) / language_folder

    if not folder_path.exists():
        raise FileNotFoundError(f"Language folder not found: {folder_path}")

    # Find CSV file in the folder
    csv_files = list(folder_path.glob("*.csv"))

    if not csv_files:
        raise FileNotFoundError(f"No CSV file found in {folder_path}")

    if len(csv_files) > 1:
        print(f"Multiple CSV files found in {folder_path}, using: {csv_files[0]}")

    csv_file = csv_files[0]

    try:
        df = pd.read_csv(csv_file, encoding='utf-8')
        print(f"Loaded {csv_file}")
    except Exception as e:
        try:
            df = pd.read_csv(csv_file, sep='\t', encoding='utf-8')
            print(f"Loaded {csv_file} (tab-separated)")
        except Exception as e2:
            raise Exception(f"Failed to load {csv_file}: {e}, {e2}")

    print(f"Original data shape: {df.shape}")
    print(f"Columns: {list(df.columns)}")

    # Verify required columns exist
    required_columns = ['filename', 'text', 'hallucination', 'hallucination_type', 'hallucination_level']
    missing_columns = [col for col in required_columns if col not in df.columns]
    if missing_columns:
        raise ValueError(f"Missing required columns in {csv_file}: {missing_columns}")

    # Apply filtering if requested
    if filter_hallucinated_only:
        filtered_df = df[df['hallucination'].str.lower() == 'yes'].copy()
        print(f"Filtered data (hallucination=yes): {filtered_df.shape}")
        processed_df = filtered_df
    else:
        print(f"Processing all data: {df.shape}")
        processed_df = df.copy()

    # Add language and folder path columns
    processed_df['language'] = language
    processed_df['language_folder'] = language_folder

    # Reset index to ensure consistent indexing
    processed_df.reset_index(drop=True, inplace=True)

    return processed_df

def load_all_transcription_data(filter_hallucinated_only: bool = False) -> Dict[str, pd.DataFrame]:
    """Load all transcription files with optional hallucination filtering"""
    all_data = {}

    for language, folder_name in LANGUAGE_FOLDERS.items():
        filter_msg = "hallucinated samples only" if filter_hallucinated_only else "all samples"
        print(f"\nLoading {language} transcriptions from {folder_name}/ ({filter_msg})...")
        try:
            data = load_transcription_data(folder_name, language, filter_hallucinated_only)
            all_data[language] = data
        except Exception as e:
            print(f"Failed to load {language} data: {e}")

    return all_data


# ========================== SMART CHECKPOINT MANAGER ==========================

class SmartCheckpointManager:
    def __init__(self, model_name: str, language: str, experiment_type: str, output_dir: str = OUTPUT_DIR):
        self.model_name = model_name.split('/')[-1]
        self.language = language
        self.experiment_type = experiment_type
        self.output_dir = Path(output_dir) / self.model_name / experiment_type
        self.checkpoint_dir = Path(CHECKPOINT_DIR) / self.model_name / experiment_type

        # Create directories
        self.output_dir.mkdir(parents=True, exist_ok=True)
        self.checkpoint_dir.mkdir(parents=True, exist_ok=True)

        # File paths
        self.results_file = self.output_dir / f"{language}_results.csv"
        self.checkpoint_file = self.checkpoint_dir / f"{language}_checkpoint.pkl"
        self.progress_file = self.checkpoint_dir / f"{language}_progress.json"
        self.sample_mapping_file = self.checkpoint_dir / f"{language}_sample_mapping.pkl"

    def _create_sample_key(self, row: pd.Series) -> str:
        filename = str(row.get('filename', ''))
        text = str(row.get('text', ''))[:100]
        language = str(row.get('language', ''))
        key_string = f"{filename}|{text}|{language}"
        return hashlib.md5(key_string.encode('utf-8')).hexdigest()

    def _create_sample_mapping(self, df: pd.DataFrame) -> Dict[str, int]:
        mapping = {}
        for idx, row in df.iterrows():
            key = self._create_sample_key(row)
            mapping[key] = idx
        return mapping

    def _load_existing_sample_mapping(self) -> Dict[str, Dict]:
        if self.sample_mapping_file.exists():
            try:
                with open(self.sample_mapping_file, 'rb') as f:
                    return pickle.load(f)
            except Exception as e:
                print(f"Warning: Failed to load sample mapping: {e}")
        return {}

    def _save_sample_mapping(self, mapping: Dict[str, Dict]):
        try:
            with open(self.sample_mapping_file, 'wb') as f:
                pickle.dump(mapping, f)
        except Exception as e:
            print(f"Warning: Failed to save sample mapping: {e}")

    def smart_load_existing_results(self, current_df: pd.DataFrame) -> Tuple[pd.DataFrame, Set[int]]:
        results_df = self._initialize_results_df(current_df)
        processed_indices = set()
        current_mapping = self._create_sample_mapping(current_df)
        print(f"Current dataset has {len(current_mapping)} unique samples")

        if not self._check_prompt_version_compatible():
            print(f"Prompt version changed (now v{PROMPT_VERSION}) - old results incompatible, starting fresh")
            return results_df, processed_indices

        if not self.results_file.exists():
            print("No existing results found - starting fresh")
            return results_df, processed_indices

        try:
            existing_df = pd.read_csv(self.results_file, encoding='utf-8')
            print(f"Found existing results with {len(existing_df)} samples")
            matched_samples = 0
            unmatched_samples = 0

            for existing_idx, existing_row in existing_df.iterrows():
                existing_key = self._create_sample_key(existing_row)
                if existing_key in current_mapping:
                    current_idx = current_mapping[existing_key]
                    pred_columns = [col for col in existing_df.columns if col.startswith('pred_')]
                    is_processed = self._check_sample_processed(existing_row, pred_columns)
                    if is_processed:
                        for col in pred_columns:
                            if col in existing_df.columns:
                                results_df.loc[current_idx, col] = existing_row[col]
                        processed_indices.add(current_idx)
                        matched_samples += 1
                    else:
                        unmatched_samples += 1
                else:
                    unmatched_samples += 1

            print(f"Successfully matched {matched_samples} processed samples")
            print(f"Found {unmatched_samples} samples that couldn't be matched or weren't processed")

            updated_mapping = {}
            for idx, row in results_df.iterrows():
                if idx in processed_indices:
                    key = self._create_sample_key(row)
                    pred_data = {}
                    for col in results_df.columns:
                        if col.startswith('pred_'):
                            pred_data[col] = results_df.loc[idx, col]
                    updated_mapping[key] = {'index': idx, 'predictions': pred_data, 'processed': True}
            self._save_sample_mapping(updated_mapping)

        except Exception as e:
            print(f"Error loading existing results: {e}")
            print("Starting fresh...")

        return results_df, processed_indices

    def _check_prompt_version_compatible(self) -> bool:
        if self.progress_file.exists():
            try:
                with open(self.progress_file, 'r') as f:
                    progress = json.load(f)
                return progress.get('prompt_version', 1) == PROMPT_VERSION
            except Exception:
                pass
        return True

    def _check_sample_processed(self, row: pd.Series, pred_columns: List[str]) -> bool:
        required_cols = [
            'pred_direct_binary', 'pred_direct_type', 'pred_direct_degree',
            'pred_cot_binary', 'pred_cot_type', 'pred_cot_degree'
        ]
        for col in required_cols:
            if col not in pred_columns:
                return False
            value = row.get(col, '')
            if pd.isna(value) or str(value).strip() == '' or str(value).lower() == 'nan':
                return False
        return True

    def save_checkpoint(self, df: pd.DataFrame, processed_indices: set, current_idx: int):
        checkpoint_data = {
            'processed_indices': processed_indices,
            'current_idx': current_idx,
            'timestamp': datetime.now().isoformat(),
            'total_samples': len(df),
            'experiment_type': self.experiment_type,
            'smart_resume': True,
            'prompt_version': PROMPT_VERSION
        }
        with open(self.checkpoint_file, 'wb') as f:
            pickle.dump(checkpoint_data, f)

        progress_info = {
            'model': self.model_name, 'language': self.language,
            'experiment_type': self.experiment_type,
            'processed_count': len(processed_indices), 'total_count': len(df),
            'progress_percentage': (len(processed_indices) / len(df)) * 100 if len(df) > 0 else 0,
            'last_updated': datetime.now().isoformat(),
            'smart_resume_enabled': True, 'prompt_version': PROMPT_VERSION
        }
        with open(self.progress_file, 'w') as f:
            json.dump(progress_info, f, indent=2)

        df.to_csv(self.results_file, index=False, encoding='utf-8')

        sample_mapping = {}
        for idx in processed_indices:
            if idx < len(df):
                key = self._create_sample_key(df.iloc[idx])
                pred_data = {col: df.loc[idx, col] for col in df.columns if col.startswith('pred_')}
                sample_mapping[key] = {'index': idx, 'predictions': pred_data, 'processed': True}
        self._save_sample_mapping(sample_mapping)

    def load_checkpoint(self) -> Tuple[Optional[set], Optional[int]]:
        if self.checkpoint_file.exists():
            try:
                with open(self.checkpoint_file, 'rb') as f:
                    checkpoint_data = pickle.load(f)
                if checkpoint_data.get('prompt_version', 1) != PROMPT_VERSION:
                    return set(), 0
                processed_indices = checkpoint_data.get('processed_indices', set())
                current_idx = checkpoint_data.get('current_idx', 0)
                if checkpoint_data.get('smart_resume', False):
                    print(f"Resuming smart checkpoint: {len(processed_indices)} samples already processed")
                    return processed_indices, current_idx
                return set(), 0
            except Exception as e:
                print(f"Failed to load checkpoint: {e}")
                return set(), 0
        return set(), 0

    def _initialize_results_df(self, df: pd.DataFrame) -> pd.DataFrame:
        results_df = df.copy()
        for approach in APPROACHES:
            for task in TASKS:
                for suffix in ['', '_raw']:
                    col = f'pred_{approach}_{task}{suffix}'
                    if col not in results_df.columns:
                        results_df[col] = ''
        return results_df

    def is_sample_processed(self, df: pd.DataFrame, idx: int) -> bool:
        required_cols = [
            'pred_direct_binary', 'pred_direct_type', 'pred_direct_degree',
            'pred_cot_binary', 'pred_cot_type', 'pred_cot_degree'
        ]
        for col in required_cols:
            if col not in df.columns:
                return False
            value = df.loc[idx, col]
            if pd.isna(value) or str(value).strip() == '' or str(value).lower() == 'nan':
                return False
        return True

    def cleanup_checkpoint(self):
        for fp in [self.checkpoint_file, self.progress_file, self.sample_mapping_file]:
            if fp.exists():
                fp.unlink()
        print(f"Cleaned up checkpoint files for {self.language} ({self.experiment_type})")

    def get_smart_resume_stats(self, current_df: pd.DataFrame) -> Dict:
        current_mapping = self._create_sample_mapping(current_df)
        existing_mapping = self._load_existing_sample_mapping()
        return {
            'current_dataset_size': len(current_df),
            'current_unique_samples': len(current_mapping),
            'existing_processed_samples': len(existing_mapping),
            'potentially_matchable': len(set(current_mapping.keys()) & set(existing_mapping.keys()))
        }


# ========================== CLASSIFIER CLASS ==========================

class HallucinationClassifier:
    def __init__(self, model_name: str, experiment_type: str, use_flash_attn: bool = False):
        self.model, self.processor = setup_model(model_name, use_flash_attn)
        self.model_name = model_name.split('/')[-1]
        self.experiment_type = experiment_type
        self.sampling_rate = self.processor.feature_extractor.sampling_rate
        # Resolve device once (device_map="auto" can make model.device unreliable)
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    def _prepare_conversation_audio(self, prompt: str, audio_path: str) -> Tuple[List[Dict], List]:
        conversation = [
            {"role": "system", "content": "You are a helpful assistant."},
            {"role": "user", "content": [
                {"type": "audio", "audio_url": audio_path},
                {"type": "text", "text": prompt}
            ]}
        ]
        audio_data, _ = librosa.load(audio_path, sr=self.sampling_rate)
        return conversation, [audio_data]

    def _prepare_conversation_text(self, prompt: str, text_content: str) -> Tuple[List[Dict], List]:
        conversation = [
            {"role": "system", "content": "You are a helpful assistant."},
            {"role": "user", "content": [
                {"type": "text", "text": f"{prompt}\n\nText to analyze: {text_content}"}
            ]}
        ]
        return conversation, []

    # =================== SINGLE-SAMPLE GENERATION ===================

    def _generate_response(self, conversation: List[Dict], audios: List) -> str:
        """Generate response for a SINGLE sample (used by audio path and OOM fallback)"""
        try:
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

            text = self.processor.apply_chat_template(
                conversation, add_generation_prompt=True, tokenize=False
            )

            if audios:
                inputs = self.processor(
                    text=text, audios=audios, return_tensors="pt", padding=True
                )
            else:
                inputs = self.processor(
                    text=text, return_tensors="pt", padding=True
                )

            inputs = inputs.to(self.device)
            input_len = inputs.input_ids.shape[1]

            with torch.no_grad():
                generate_ids = self.model.generate(
                    **inputs, max_new_tokens=256, do_sample=False,
                )

            del inputs
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

            generate_ids = generate_ids[:, input_len:]
            response = self.processor.batch_decode(
                generate_ids, skip_special_tokens=True,
                clean_up_tokenization_spaces=False
            )[0]

            del generate_ids
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

            return response.strip()

        except torch.cuda.OutOfMemoryError as e:
            print(f"CUDA Out of Memory: {e}")
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
                torch.cuda.synchronize()
            time.sleep(2)
            return ""

        except Exception as e:
            print(f"Error generating response: {e}")
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
            return ""

    # =================== BATCHED TEXT GENERATION ===================

    def _generate_response_batch_text(self, conversations: List[List[Dict]]) -> List[str]:
        """
        Generate responses for a BATCH of text-only conversations.
        Falls back to single-sample processing on OOM.
        """
        if not conversations:
            return []

        try:
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

            # Tokenize all conversations
            texts = [
                self.processor.apply_chat_template(
                    conv, add_generation_prompt=True, tokenize=False
                )
                for conv in conversations
            ]

            # Text-only: no audios needed
            inputs = self.processor(
                text=texts, return_tensors="pt", padding=True
            )

            inputs = inputs.to(self.device)
            input_len = inputs.input_ids.shape[1]

            with torch.no_grad():
                generate_ids = self.model.generate(
                    **inputs, max_new_tokens=256, do_sample=False,
                )

            del inputs
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

            # Slice off the input tokens to get only generated tokens
            generate_ids = generate_ids[:, input_len:]

            # Decode all responses
            responses = self.processor.batch_decode(
                generate_ids, skip_special_tokens=True,
                clean_up_tokenization_spaces=False
            )

            del generate_ids
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

            return [r.strip() for r in responses]

        except torch.cuda.OutOfMemoryError as e:
            print(f"\nCUDA OOM on batch of {len(conversations)}. Falling back to single-sample...")
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
                torch.cuda.synchronize()
            time.sleep(2)

            # Fallback: process one at a time
            responses = []
            for conv in conversations:
                resp = self._generate_response(conv, [])
                responses.append(resp)
            return responses

        except Exception as e:
            print(f"\nBatch generation error: {e}. Falling back to single-sample...")
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

            responses = []
            for conv in conversations:
                resp = self._generate_response(conv, [])
                responses.append(resp)
            return responses

    def classify_task_batch_text(self, text_inputs: List[str], task: str, approach: str) -> List[Dict[str, str]]:
        """
        Run a single classification task on a BATCH of text inputs.
        Returns list of dicts with 'value' and 'raw_response'.
        """
        prompt = PROMPTS['text'][task][approach]

        # Build conversations for entire batch
        conversations = []
        for text_content in text_inputs:
            conv, _ = self._prepare_conversation_text(prompt, text_content)
            conversations.append(conv)

        # Batch inference
        responses = self._generate_response_batch_text(conversations)

        # Parse each response
        results = []
        for response in responses:
            if task == 'binary':
                value = self._parse_binary(response)
            elif task == 'type':
                value = self._parse_type(response)
            elif task == 'degree':
                value = self._parse_degree(response)
            else:
                value = 'none'
            results.append({'value': value, 'raw_response': response})

        return results

    # =================== SINGLE-SAMPLE CLASSIFY (unchanged) ===================

    def classify_task(self, input_data: str, task: str, approach: str) -> Dict[str, str]:
        """Run a single classification task with a given approach (single sample)."""
        prompt = PROMPTS[self.experiment_type][task][approach]

        if self.experiment_type == 'audio':
            conversation, audios = self._prepare_conversation_audio(prompt, input_data)
        else:
            conversation, audios = self._prepare_conversation_text(prompt, input_data)

        response = self._generate_response(conversation, audios)

        if task == 'binary':
            value = self._parse_binary(response)
        elif task == 'type':
            value = self._parse_type(response)
        elif task == 'degree':
            value = self._parse_degree(response)
        else:
            value = 'none'

        return {'value': value, 'raw_response': response}

    # =================== PARSERS ===================

    def _parse_binary(self, response: str) -> str:
        if not response:
            return 'no'
        try:
            start = response.find('{')
            end = response.rfind('}') + 1
            if start != -1 and end != 0:
                result = json.loads(response[start:end])
                binary = result.get('binary', 'no').lower().strip()
                if binary in ('yes', 'no'):
                    return binary
        except (json.JSONDecodeError, AttributeError):
            pass
        response_lower = response.lower()
        if any(phrase in response_lower for phrase in ['"binary": "yes"', "'binary': 'yes'", 'binary: yes', '"yes"']):
            return 'yes'
        return 'no'

    def _parse_type(self, response: str) -> str:
        if not response:
            return 'none'
        valid_types = {'factual_contradiction', 'factual_fabrication', 'contextual_inconsistency', 'none'}
        try:
            start = response.find('{')
            end = response.rfind('}') + 1
            if start != -1 and end != 0:
                result = json.loads(response[start:end])
                type_val = result.get('type', 'none').lower().strip()
                if type_val in valid_types:
                    return type_val
        except (json.JSONDecodeError, AttributeError):
            pass
        response_lower = response.lower()
        if 'factual_fabrication' in response_lower:
            return 'factual_fabrication'
        elif 'factual_contradiction' in response_lower:
            return 'factual_contradiction'
        elif 'contextual_inconsistency' in response_lower or 'contextual inconsistency' in response_lower:
            return 'contextual_inconsistency'
        elif 'fabrication' in response_lower:
            return 'factual_fabrication'
        elif 'contradiction' in response_lower:
            return 'factual_contradiction'
        return 'none'

    def _parse_degree(self, response: str) -> str:
        if not response:
            return 'none'
        valid_degrees = {'mild', 'moderate', 'severe', 'none'}
        try:
            start = response.find('{')
            end = response.rfind('}') + 1
            if start != -1 and end != 0:
                result = json.loads(response[start:end])
                degree = result.get('degree', 'none').lower().strip()
                if degree in valid_degrees:
                    return degree
        except (json.JSONDecodeError, AttributeError):
            pass
        response_lower = response.lower()
        if 'severe' in response_lower:
            return 'severe'
        elif 'moderate' in response_lower:
            return 'moderate'
        elif 'mild' in response_lower:
            return 'mild'
        return 'none'

# ========================== EXPERIMENT RUNNER ==========================

def run_experiment_with_smart_resume(model_name, data_dict, experiment_type, force_restart=False, use_flash_attn=False):
    """Run classification experiment with smart checkpointing.
    Uses batched inference for text experiments."""

    batch_size = TEXT_BATCH_SIZE if experiment_type == 'text' else 1

    print(f"\nStarting {experiment_type} experiment with {model_name} (Smart Resume Enabled)")
    print(f"Prompt version: v{PROMPT_VERSION} (separated single-task prompts)")
    print(f"Tasks: {TASKS} x Approaches: {APPROACHES} = {len(TASKS) * len(APPROACHES)} calls per sample")
    print(f"Batch size: {batch_size}" + (" (text batching enabled)" if batch_size > 1 else " (single sample)"))
    print("=" * 60)

    print_gpu_memory_info()

    # Initialize classifier
    classifier = HallucinationClassifier(model_name, experiment_type, use_flash_attn)

    print_gpu_memory_info()

    # Process each language
    for language, df in data_dict.items():
        print(f"\nProcessing {language} data ({len(df)} samples) - {experiment_type} mode (batch_size={batch_size})")

        checkpoint_manager = SmartCheckpointManager(model_name, language, experiment_type)

        smart_stats = checkpoint_manager.get_smart_resume_stats(df)
        print(f"Dataset info: {smart_stats['current_dataset_size']} total, {smart_stats['current_unique_samples']} unique")
        print(f"Resume info: {smart_stats['existing_processed_samples']} previously processed, {smart_stats['potentially_matchable']} matchable")

        if force_restart:
            print("Force restart - ignoring existing results")
            results_df = checkpoint_manager._initialize_results_df(df)
            processed_indices = set()
        else:
            print("Using smart resume to match existing results...")
            results_df, processed_indices = checkpoint_manager.smart_load_existing_results(df)

        print(f"Smart resume result: {len(processed_indices)}/{len(df)} samples already processed")

        if len(processed_indices) == len(df):
            print(f"All samples for {language} ({experiment_type}) already processed!")
            continue

        # Collect unprocessed indices
        unprocessed_indices = [idx for idx in range(len(df)) if idx not in processed_indices]
        print(f"Samples remaining: {len(unprocessed_indices)}")

        pbar = tqdm(total=len(df), initial=len(processed_indices),
                   desc=f"Processing {language} ({experiment_type})", unit="samples")

        samples_since_checkpoint = 0
        checkpoint_interval = max(5, batch_size * 2)

        try:
            # ---- TEXT EXPERIMENT: BATCHED PROCESSING ----
            if experiment_type == 'text' and batch_size > 1:
                for batch_start in range(0, len(unprocessed_indices), batch_size):
                    batch_indices = unprocessed_indices[batch_start:batch_start + batch_size]

                    # Gather valid text inputs
                    valid_batch = []
                    for idx in batch_indices:
                        row = df.iloc[idx]
                        if pd.isna(row['text']) or str(row['text']).strip() == '':
                            print(f"\nEmpty text for sample {idx}, skipping")
                            pbar.update(1)
                            continue
                        valid_batch.append((idx, str(row['text'])))

                    if not valid_batch:
                        continue

                    batch_idx_list = [item[0] for item in valid_batch]
                    batch_texts = [item[1] for item in valid_batch]

                    # Memory info periodically
                    if batch_start % (batch_size * 25) == 0 and torch.cuda.is_available():
                        mem_alloc = torch.cuda.memory_allocated() / 1024**3
                        mem_res = torch.cuda.memory_reserved() / 1024**3
                        print(f"\nGPU Memory: {mem_alloc:.1f}GB allocated, {mem_res:.1f}GB reserved")

                    # Run all 6 task/approach combos in batches
                    for approach in APPROACHES:
                        for task in TASKS:
                            batch_results = classifier.classify_task_batch_text(
                                batch_texts, task, approach
                            )
                            for i, idx in enumerate(batch_idx_list):
                                results_df.loc[idx, f'pred_{approach}_{task}'] = batch_results[i]['value']
                                results_df.loc[idx, f'pred_{approach}_{task}_raw'] = batch_results[i]['raw_response']

                            time.sleep(0.1)
                            if torch.cuda.is_available():
                                torch.cuda.empty_cache()

                    # Mark batch as processed
                    for idx in batch_idx_list:
                        processed_indices.add(idx)
                    samples_since_checkpoint += len(batch_idx_list)
                    pbar.update(len(batch_idx_list))

                    if samples_since_checkpoint >= checkpoint_interval:
                        checkpoint_manager.save_checkpoint(results_df, processed_indices, batch_idx_list[-1])
                        samples_since_checkpoint = 0

                    if torch.cuda.is_available():
                        torch.cuda.empty_cache()
                    time.sleep(0.1)

            # ---- AUDIO EXPERIMENT OR TEXT batch_size=1: SINGLE-SAMPLE ----
            else:
                for idx in unprocessed_indices:
                    row = df.iloc[idx]
                    try:
                        if experiment_type == 'audio':
                            audio_filename = row['filename']
                            language_folder = row['language_folder']
                            audio_path = os.path.join(BASE_DATA_DIR, language_folder, audio_filename)
                            if not os.path.exists(audio_path):
                                print(f"\nAudio file not found: {audio_path}")
                                pbar.update(1)
                                continue
                            input_data = audio_path
                        else:
                            if pd.isna(row['text']) or row['text'].strip() == '':
                                print(f"\nEmpty text for sample {idx}")
                                pbar.update(1)
                                continue
                            input_data = row['text']

                        if idx % 100 == 0 and torch.cuda.is_available():
                            mem_alloc = torch.cuda.memory_allocated() / 1024**3
                            mem_res = torch.cuda.memory_reserved() / 1024**3
                            print(f"\nGPU Memory: {mem_alloc:.1f}GB allocated, {mem_res:.1f}GB reserved")

                        for approach in APPROACHES:
                            for task in TASKS:
                                result = classifier.classify_task(input_data, task, approach)
                                results_df.loc[idx, f'pred_{approach}_{task}'] = result['value']
                                results_df.loc[idx, f'pred_{approach}_{task}_raw'] = result['raw_response']
                                time.sleep(0.1)
                                if torch.cuda.is_available():
                                    torch.cuda.empty_cache()

                        processed_indices.add(idx)
                        samples_since_checkpoint += 1
                        pbar.update(1)

                        if samples_since_checkpoint >= checkpoint_interval:
                            checkpoint_manager.save_checkpoint(results_df, processed_indices, idx)
                            samples_since_checkpoint = 0

                        if torch.cuda.is_available():
                            torch.cuda.empty_cache()
                        time.sleep(0.2)

                    except Exception as e:
                        print(f"\nError processing sample {idx}: {e}")
                        if torch.cuda.is_available():
                            torch.cuda.empty_cache()
                        pbar.update(1)
                        continue

        except KeyboardInterrupt:
            print(f"\nInterrupted! Saving progress...")
            checkpoint_manager.save_checkpoint(results_df, processed_indices,
                batch_idx_list[-1] if 'batch_idx_list' in locals() else 0)
            print(f"Progress saved. You can resume later.")
            return

        finally:
            pbar.close()

        # Final save
        checkpoint_manager.save_checkpoint(results_df, processed_indices, len(df)-1)

        summary = generate_summary_stats(results_df, language, classifier.model_name, experiment_type)
        summary_file = checkpoint_manager.output_dir / f"{language}_summary.json"
        with open(summary_file, 'w', encoding='utf-8') as f:
            json.dump(summary, f, indent=2, ensure_ascii=False)

        print(f"Completed {language} ({experiment_type}): {len(processed_indices)}/{len(df)} samples")
        print(f"Results saved to: {checkpoint_manager.results_file}")
        print(f"Summary saved to: {summary_file}")

        if len(processed_indices) == len(df):
            checkpoint_manager.cleanup_checkpoint()

def generate_summary_stats(df: pd.DataFrame, language: str, model_name: str, experiment_type: str) -> Dict:
    summary = {
        'language': language, 'model': model_name,
        'experiment_type': experiment_type, 'total_samples': len(df),
        'prompt_version': PROMPT_VERSION, 'timestamp': time.strftime('%Y-%m-%d %H:%M:%S')
    }
    for approach in APPROACHES:
        binary_col = f'pred_{approach}_binary'
        if binary_col in df.columns:
            binary_yes = (df[binary_col] == 'yes').sum()
            type_col = f'pred_{approach}_type'
            degree_col = f'pred_{approach}_degree'
            summary[approach] = {
                'binary_yes_predictions': int(binary_yes),
                'binary_accuracy': float(binary_yes / len(df)) if len(df) > 0 else 0,
                'type_distribution': df[type_col].value_counts().to_dict() if type_col in df.columns else {},
                'degree_distribution': df[degree_col].value_counts().to_dict() if degree_col in df.columns else {}
            }
    return summary


# ========================== VALIDATION AND UTILITIES ==========================

def validate_smart_resume(model_name: str, language: str, experiment_type: str):
    print(f"Validating smart resume for {model_name} - {language} - {experiment_type}")
    print("-" * 50)
    data_dict_old = load_all_transcription_data(filter_hallucinated_only=True)
    data_dict_new = load_all_transcription_data(filter_hallucinated_only=False)
    if language not in data_dict_old or language not in data_dict_new:
        print(f"Language {language} not found in datasets")
        return 0, 0
    old_df, new_df = data_dict_old[language], data_dict_new[language]
    print(f"Old dataset: {len(old_df)} | New dataset: {len(new_df)}")
    cm = SmartCheckpointManager(model_name, language, experiment_type)
    old_map, new_map = cm._create_sample_mapping(old_df), cm._create_sample_mapping(new_df)
    matches = set(old_map.keys()) & set(new_map.keys())
    print(f"Matches: {len(matches)}/{len(old_map)} ({len(matches)/len(old_map)*100:.1f}%)")
    if len(matches) < len(old_map) * 0.9:
        print("WARNING: Low match rate.")
    else:
        print("Good match rate!")
    return len(matches), len(old_map)

def show_progress_summary():
    print("EXPERIMENT PROGRESS SUMMARY")
    print("=" * 60)
    checkpoint_base = Path(CHECKPOINT_DIR)
    if not checkpoint_base.exists():
        print("No experiments in progress.")
        return
    for model_dir in checkpoint_base.iterdir():
        if not model_dir.is_dir(): continue
        print(f"\nModel: {model_dir.name}")
        for exp_dir in model_dir.iterdir():
            if not exp_dir.is_dir(): continue
            print(f"  {exp_dir.name}:")
            for pf in exp_dir.glob("*_progress.json"):
                try:
                    with open(pf) as f: p = json.load(f)
                    print(f"    {p.get('language','?'):10} | {p.get('processed_count',0)}/{p.get('total_count',0)} ({p.get('progress_percentage',0):.1f}%) | v{p.get('prompt_version','?')}")
                except Exception as e:
                    print(f"    Error: {e}")

def print_gpu_memory_info():
    if torch.cuda.is_available():
        a = torch.cuda.memory_allocated() / 1024**3
        r = torch.cuda.memory_reserved() / 1024**3
        t = torch.cuda.get_device_properties(0).total_memory / 1024**3
        print(f"GPU Memory: {a:.1f}GB/{t:.1f}GB used ({r:.1f}GB reserved, {t-a:.1f}GB free)")
    else:
        print("CUDA not available")

def merge_all_results():
    for model_dir in Path(OUTPUT_DIR).iterdir():
        if not model_dir.is_dir(): continue
        print(f"Merging results for {model_dir.name}")
        for exp_dir in model_dir.iterdir():
            if not exp_dir.is_dir(): continue
            dfs = [pd.read_csv(f, encoding='utf-8') for f in exp_dir.glob("*_results.csv")]
            if dfs:
                merged = pd.concat(dfs, ignore_index=True)
                out = exp_dir / "all_languages_results.csv"
                merged.to_csv(out, index=False, encoding='utf-8')
                print(f"  Saved: {out}")


# ========================== MAIN EXECUTION ==========================

def main_with_smart_resume(experiment_types: List[str] = None, force_restart: bool = False,
                          use_flash_attn: bool = False, filter_hallucinated_only: bool = False,
                          validate_first: bool = True):

    print("Hallucination Classification Experiment with Smart Resume")
    print(f"Prompt version: v{PROMPT_VERSION} | Text batch size: {TEXT_BATCH_SIZE}")
    print("=" * 60)

    if experiment_types is None:
        experiment_types = EXPERIMENT_TYPES
    valid_types = [t for t in experiment_types if t in EXPERIMENT_TYPES]
    if not valid_types:
        print(f"Invalid experiment types. Valid: {EXPERIMENT_TYPES}")
        return

    print(f"Experiment types: {valid_types}")
    print(f"Data mode: {'Hallucinated only' if filter_hallucinated_only else 'All samples'}")

    print("\nLoading transcription data...")
    data_dict = load_all_transcription_data(filter_hallucinated_only=filter_hallucinated_only)
    if not data_dict:
        print("No data loaded.")
        return

    total = sum(len(df) for df in data_dict.values())
    print(f"\nTotal samples: {total}")
    for lang, df in data_dict.items():
        if 'hallucination' in df.columns:
            h = (df['hallucination'].str.lower() == 'yes').sum()
            print(f"  - {lang}: {len(df)} (hallucinated: {h}, non-hallucinated: {len(df)-h})")
        else:
            print(f"  - {lang}: {len(df)}")

    if validate_first and not force_restart and not filter_hallucinated_only:
        print("\nValidating smart resume...")
        for mn in MODELS:
            for et in valid_types:
                for lang in data_dict:
                    try:
                        m, t = validate_smart_resume(mn, lang, et)
                        if m < t * 0.9:
                            resp = input(f"Low match for {mn}-{lang}-{et}. Continue? (y/n): ")
                            if resp.lower() != 'y': return
                    except Exception as e:
                        print(f"Validation failed: {e}")

    Path(OUTPUT_DIR).mkdir(exist_ok=True)
    Path(CHECKPOINT_DIR).mkdir(exist_ok=True)

    for model_name in MODELS:
        for experiment_type in valid_types:
            try:
                run_experiment_with_smart_resume(model_name, data_dict, experiment_type, force_restart, use_flash_attn)
            except Exception as e:
                import traceback
                print(f"Failed: {experiment_type} with {model_name}: {e}")
                traceback.print_exc()
                continue

    print(f"\nAll experiments completed! Results in: {OUTPUT_DIR}")


# ========================== EXECUTION ==========================

if __name__ == "__main__":
    main_with_smart_resume(
        experiment_types=['text'],       # <-- text only for batched run
        force_restart=False,
        use_flash_attn=False,
        filter_hallucinated_only=False,
        validate_first=True
    )

    # Other options:
    # main_with_smart_resume(experiment_types=['audio'], ...)  # audio still batch_size=1
    # main_with_smart_resume(experiment_types=['audio', 'text'], ...)
    # show_progress_summary()
    # merge_all_results()

Hallucination Classification Experiment with Smart Resume
Prompt version: v2 | Text batch size: 8
Experiment types: ['text']
Data mode: All samples

Loading transcription data...

Loading english transcriptions from English/ (all samples)...
Loaded /content/drive/MyDrive/ICASSP_Hallucinaton/Audio_data/English/metadata.csv
Original data shape: (3966, 9)
Columns: ['id', 'speaker', 'accent', 'gender', 'filename', 'text', 'hallucination', 'hallucination_type', 'hallucination_level']
Processing all data: (3966, 9)

Loading kazakh transcriptions from Kazakh/ (all samples)...
Loaded /content/drive/MyDrive/ICASSP_Hallucinaton/Audio_data/Kazakh/metadata.csv
Original data shape: (3978, 8)
Columns: ['id', 'voice', 'gender', 'filename', 'text', 'hallucination', 'hallucination_type', 'hallucination_level']
Processing all data: (3978, 8)

Loading russian transcriptions from Russian/ (all samples)...
Loaded /content/drive/MyDrive/ICASSP_Hallucinaton/Audio_data/Russian/metadata.csv
Original data shape

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/876 [00:00<?, ?it/s]

Model loaded successfully: Qwen/Qwen2-Audio-7B-Instruct
GPU Memory: 15.6GB/95.0GB used (71.0GB reserved, 79.3GB free)

Processing english data (3966 samples) - text mode (batch_size=8)
Dataset info: 3966 total, 3966 unique
Resume info: 3965 previously processed, 3965 matchable
Using smart resume to match existing results...
Current dataset has 3966 unique samples
Found existing results with 3966 samples
Successfully matched 3965 processed samples
Found 1 samples that couldn't be matched or weren't processed
Smart resume result: 3965/3966 samples already processed
Samples remaining: 1




Processing english (text): 100%|██████████| 3966/3966 [00:00<00:00, 9000.65samples/s]

...


GPU Memory: 15.6GB allocated, 15.7GB reserved




Processing russian (text):  95%|█████████▍| 3846/4068 [1:58:12<08:15,  2.23s/samples]

Processing russian (text):  95%|█████████▍| 3854/4068 [1:58:31<08:04,  2.26s/samples]

Processing russian (text):  95%|█████████▍| 3862/4068 [1:58:50<07:56,  2.31s/samples]

Processing russian (text):  95%|█████████▌| 3870/4068 [1:59:09<07:41,  2.33s/samples]

Processing russian (text):  95%|█████████▌| 3878/4068 [1:59:37<08:29,  2.68s/samples]

Processing russian (text):  96%|█████████▌| 3886/4068 [1:59:55<07:42,  2.54s/samples]

Processing russian (text):  96%|█████████▌| 3894/4068 [2:00:51<11:17,  3.89s/samples]

Processing russian (text):  96%|█████████▌| 3902/4068 [2:01:31<11:42,  4.23s/samples]

Processing russian (text):  96%|█████████▌| 3910/4068 [2:02:23<12:54,  4.90s/samples]

Processing russian (text):  96%|█████████▋| 3918/4068 [2:03:43<16:05,  6.44s/samples]

Processing russian (text):  97%|█████████▋| 3926/4068 [2:05:19<19:08,  8.09s/samples]

Processing russian (text):  97%|█████████


GPU Memory: 15.6GB allocated, 15.7GB reserved




Processing russian (text):  99%|█████████▉| 4046/4068 [2:11:47<00:49,  2.27s/samples]

Processing russian (text): 100%|█████████▉| 4054/4068 [2:12:05<00:31,  2.25s/samples]

Processing russian (text): 100%|█████████▉| 4062/4068 [2:12:22<00:13,  2.23s/samples]

Processing russian (text): 100%|██████████| 4068/4068 [2:12:39<00:00,  3.03s/samples]


Completed russian (text): 4067/4068 samples
Results saved to: hallucination_results/Qwen2-Audio-7B-Instruct/text/russian_results.csv
Summary saved to: hallucination_results/Qwen2-Audio-7B-Instruct/text/russian_summary.json

All experiments completed! Results in: hallucination_results
